# Wi-Fi 7 multi-link EDCA — the manuscript's figures

Reproduces **Fig. 2, Fig. 3, Fig. 4 and Table I** from the stored measurements.
Nothing is re-simulated: every number was produced by the analytical model in
the main repository and written to the JSON files loaded below.

**Data files** (upload them, or put them in a folder named `data` beside the
notebook):

| file | used by |
|---|---|
| `pp1_paper_figs.json` | Fig. 2(a) |
| `gnn_train_curve.json` | Fig. 2(b) |
| `pp1_qos_methods.json` | Fig. 3 |
| `pp1_budget_methods.json` | Fig. 4 |
| `benchmark.json` | Table I, upper block |
| `pipeline_ngen1.json` | Table I, upper block |
| `pp1_anytime.json` | Table I, lower block |

Three configurations recur and are **cumulative** — each adds one component to
the one above:

| key | label | what it is |
|---|---|---|
| `ga` | GA baseline (Yi et al.) | genetic search on the analytical model |
| `ga_gnn` | `+` evaluation GNN $g_\phi$ | GNN predicting $(\log_{10} c_i,\ \theta_i)$; **ranks** candidates so the analytical model is called less |
| `ga_gnn_policy` | `+` proposal GNN $\pi_\psi$ | policy GNN mapping a scenario to a distribution over gene levels; **seeds** the initial population |

Run the cells in order.

In [ ]:
# --- Cell 1: locate the data -------------------------------------------------
import json, os, glob

NEEDED = ["pp1_paper_figs.json", "gnn_train_curve.json", "pp1_qos_methods.json",
          "pp1_budget_methods.json", "benchmark.json", "pipeline_ngen1.json",
          "pp1_anytime.json"]

SEARCH = ["data", ".", "/content/data", "/content",
          "/content/drive/MyDrive/data", "/content/wifi7_colab_figures/data"]

def find_data_dir():
    # Also look a level or two down: a zip dropped into Colab unpacks into a
    # folder of its own, and Drive mounts under a path of its own.
    cands = list(SEARCH)
    for base in (".", "/content"):
        if os.path.isdir(base):
            cands += sorted(glob.glob(os.path.join(base, "*", "data"))
                            + glob.glob(os.path.join(base, "*")))
    for d in cands:
        if os.path.isdir(d) and all(
                os.path.exists(os.path.join(d, f)) for f in NEEDED):
            return d
    return None

def unpack_any_zip():
    # One zip is easier to upload than seven files; accept either.
    import zipfile
    for z in glob.glob("*.zip") + glob.glob("/content/*.zip"):
        try:
            zipfile.ZipFile(z).extractall(os.path.dirname(z) or ".")
            print("  unpacked", os.path.basename(z))
        except zipfile.BadZipFile:
            pass

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    unpack_any_zip()
    DATA_DIR = find_data_dir()

if DATA_DIR is None:
    try:
        from google.colab import files
        print("Pick the seven JSON files (Ctrl-click to select all), or the "
              "zip that contains them.")
        print("The dialog has to be completed -- cancelling it leaves nothing "
              "to load, which is the usual cause of the error below.")
        files.upload()
        unpack_any_zip()
        DATA_DIR = find_data_dir()
    except ImportError:
        pass                       # not on Colab; fall through to the message

if DATA_DIR is None:
    here = sorted(os.listdir("."))[:12]
    raise FileNotFoundError(
        "None of the searched folders holds all seven files.\n"
        "  needed  : %s\n"
        "  searched: %s\n"
        "  cwd (%s) contains: %s\n"
        "Upload the files, or the zip that holds them, and re-run this cell."
        % (", ".join(NEEDED), ", ".join(SEARCH), os.path.abspath("."),
           ", ".join(here) or "(nothing)"))

print("data dir:", os.path.abspath(DATA_DIR))
for f in NEEDED:
    print("  %-26s %8.1f kB" % (f, os.path.getsize(os.path.join(DATA_DIR, f)) / 1e3))

In [ ]:
# --- Cell 2: plotting style --------------------------------------------------
# Matched to an IEEEtran two-column layout: figures are drawn at the width they
# are printed at, so the point sizes below are read literally on the page.
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

COL, PAGE = 3.50, 7.16          # \columnwidth and \textwidth, in inches
PT_TICK, PT_LABEL, PT_TITLE, PT_LEGEND, PT_NOTE = 7.0, 8.0, 8.0, 7.0, 6.5

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Nimbus Roman No9 L", "Liberation Serif",
                   "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": PT_TICK,
    "axes.labelsize": PT_LABEL, "axes.titlesize": PT_TITLE,
    "axes.titleweight": "normal", "axes.titlepad": 3.0, "axes.labelpad": 2.0,
    "axes.linewidth": 0.6, "axes.edgecolor": "black",
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": True, "axes.spines.right": True,
    "grid.color": "#CCCCCC", "grid.linewidth": 0.4, "grid.linestyle": "-",
    "xtick.labelsize": PT_TICK, "ytick.labelsize": PT_TICK,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "xtick.major.size": 2.6, "ytick.major.size": 2.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.pad": 2.0, "ytick.major.pad": 2.0,
    "legend.frameon": True, "legend.edgecolor": "black",
    "legend.framealpha": 1.0, "legend.fancybox": False,
    "legend.fontsize": PT_LEGEND, "legend.borderpad": 0.35,
    "legend.labelspacing": 0.28, "legend.handlelength": 2.0,
    "legend.handletextpad": 0.45, "legend.columnspacing": 1.1,
    "lines.linewidth": 1.1, "lines.markersize": 3.4,
    "lines.markeredgewidth": 0.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "savefig.bbox": "tight",
    "savefig.pad_inches": 0.01, "figure.dpi": 110,
})

# Okabe-Ito: distinguishable under colour blindness and in greyscale print.
# One (colour, linestyle, marker) triple per method, fixed across every figure.
STYLE = {"ga":            {"c": "#000000", "ls": "-",  "m": "o"},
         "ga_gnn":        {"c": "#0072B2", "ls": "--", "m": "s"},
         "ga_gnn_policy": {"c": "#D55E00", "ls": "-",  "m": "^"}}
C_DEF, RED, GREY = "#7F7F7F", "#C0392B", "#555555"
ORDER = ["ga", "ga_gnn", "ga_gnn_policy"]

LAB = {"ga": "GA baseline (Yi et al.)",
       "ga_gnn": r"$+$ evaluation GNN $g_\phi$ (ranks candidates)",
       "ga_gnn_policy": r"$+$ proposal GNN $\pi_\psi$ (seeds the population)"}
LAB_S = {"ga": "GA baseline (Yi et al.)",
         "ga_gnn": r"$+\,g_\phi$ (evaluation GNN)",
         "ga_gnn_policy": r"$+\,\pi_\psi$ (proposal GNN)"}
LAB_TXT = {"ga": "GA baseline (Yi et al.)",
           "ga_gnn": "  + evaluation GNN (g_phi)",
           "ga_gnn_policy": "  + proposal GNN (pi_psi)"}
LAB_TEX = {"ga": r"GA baseline~\cite{Yi2025}",
           "ga_gnn": r"\;\; $+$ evaluation GNN $g_\phi$",
           "ga_gnn_policy": r"\;\; $+$ proposal GNN $\pi_\psi$"}

def legend_below(fig, ax, ncol=3, y=-0.16, handles=None, labels=None):
    """One shared legend under the figure, so it covers no curve."""
    if handles is None:
        handles, labels = ax.get_legend_handles_labels()
    return fig.legend(handles, labels, loc="lower center", ncol=ncol,
                      bbox_to_anchor=(0.5, y), fontsize=PT_LEGEND,
                      frameon=True, edgecolor="black", framealpha=1.0,
                      fancybox=False)

print("style ready")

In [ ]:
# --- Cell 3: load ------------------------------------------------------------
def load(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as fh:
        return json.load(fh)

D   = load("pp1_paper_figs.json")       # proposal-GNN training curve, GA runs
TC  = load("gnn_train_curve.json")      # evaluation-GNN training curve
QM  = load("pp1_qos_methods.json")      # per-AC QoS, four configurations
BM  = load("pp1_budget_methods.json")   # threshold sweep x budget x method
B   = load("benchmark.json")            # 70-run ablation at a fixed config
NG1 = load("pipeline_ngen1.json")["160x1"]
AT  = load("pp1_anytime.json")          # objective against compute

print("baseline runs at Npop = 200 :", " ".join("%.2f" % v for v in D["ga_runs"]))
print("  median %.2f   range %.2f-%.2f"
      % (D["ga_fit_med"], D["ga_fit_min"], D["ga_fit_max"]))
print("eps_1 grid                  :", BM["eps1"])
print("QoS configurations          :", list(QM["configs"]))
print("anytime run                 : %d seeds, no early stopping" % AT["n_seed"])
print("evaluation-GNN curve        : %d checkpoints, %d epochs, ceiling %.3f"
      % (len(TC["curve"]), TC["epochs"], TC["pool"]["oracle_topv"]))

## Fig. 2 — training convergence of both networks

The two networks are trained on different problems and **cannot share an x
axis**. The **y axis is** shared, and that is the point: each is scored by the
objective its own output achieves under the analytical model.

- $\pi_\psi$ emits a configuration → scored by that configuration's exact $F$.
- $g_\phi$ emits an **ordering** → scored the way the pipeline uses it: the mean
  exact $F$ of the $V = 8$ candidates it forwards for exact evaluation, from a
  pool of 1,560 configurations scored once in advance. A regression error does
  not belong on this axis; the ranking it induces does.

The baseline enters panel (a) as the **median** of its ten runs, with the range
shaded and the runs drawn individually at the right edge — not as their best.
One network run is compared with one baseline run; a best-of-$N$ has $N$ chances
where one run has one.

In [ ]:
# --- Cell 4: Fig. 2 ----------------------------------------------------------
def fig_training(save="fig2_gnn_training.png"):
    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.45), sharey=True)
    cg, cp, ce = STYLE["ga"]["c"], STYLE["ga_gnn_policy"]["c"], STYLE["ga_gnn"]["c"]

    # ---- (a) proposal network --------------------------------------------
    e1 = D["e1"]
    step = np.array(e1["step"], float)
    exact = np.array(e1["exact"], float)
    ok = np.array(e1["feasible"], bool)
    runs = np.array(D["ga_runs"], float)

    ax = axes[0]
    ax.axhspan(D["ga_fit_min"], D["ga_fit_max"], color=cg, alpha=0.06, zorder=0)
    ax.axhline(D["ga_fit_med"], color=cg, ls="--", lw=1.0, zorder=2)
    ax.plot(np.full(len(runs), 0.955), runs, "_", color=cg, ms=8, mew=1.1,
            ls="none", transform=ax.get_yaxis_transform(), zorder=4)
    ax.text(0.02, D["ga_fit_med"] - 1.4,
            "GA baseline: median (dashed), range (shaded),\neach of $10$ runs (right)",
            transform=ax.get_yaxis_transform(), ha="left", va="top",
            fontsize=PT_NOTE, color=cg, linespacing=1.25)

    ax.plot(step, exact, color=cp, lw=1.2, zorder=3, label=r"$\pi_\psi$, one run")
    ax.plot(step[ok], exact[ok], "^", color=cp, ms=3.6, mec="black", mew=0.4,
            ls="none", zorder=4, label="Feasible")
    ax.plot(step[~ok], exact[~ok], "x", color=RED, ms=3.8, mew=1.0,
            ls="none", zorder=4, label="Infeasible")
    ax.set_xlabel("Cross-entropy training step")
    ax.set_ylabel("Objective $F$ (analytical model)")
    ax.set_title(r"(a) Proposal GNN $\pi_\psi$ vs GA at $29{,}309$ exact calls")
    ax.legend(loc="lower right", fontsize=PT_NOTE, borderpad=0.3,
              labelspacing=0.22, handlelength=1.6, handletextpad=0.4)

    # ---- (b) evaluation network ------------------------------------------
    c = TC["curve"]
    ep = np.array([r["epoch"] for r in c], float)
    mean_v = np.array([r["exact"] for r in c], float)
    best_v = np.array([r["exact_best"] for r in c], float)
    ceil = TC["pool"]["oracle_topv"]

    ax = axes[1]
    ax.axhline(ceil, color=GREY, lw=0.8, ls=(0, (1, 2)), zorder=2)
    ax.text(0.98, ceil - 1.2, "perfect ordering of the same pool",
            transform=ax.get_yaxis_transform(), ha="right", va="top",
            fontsize=PT_NOTE, color=GREY)
    ax.fill_between(ep, mean_v, best_v, color=ce, alpha=0.12, lw=0)
    ax.plot(ep, best_v, ls=(0, (1, 1.6)), color=ce, lw=0.9, zorder=3,
            label=r"best of the $V = 8$ it forwards")
    ax.plot(ep, mean_v, "-s", color=ce, ms=3.2, mec="black", mew=0.4, zorder=4,
            label=r"mean of the $V = 8$ it forwards")
    ax.set_xlabel("Training epoch")
    ax.set_title(r"(b) Evaluation GNN $g_\phi$, $%d$ epochs in $%d$ s"
                 % (TC["epochs"], round(c[-1]["elapsed"])))
    ax.legend(loc="lower right", fontsize=PT_NOTE, borderpad=0.3,
              labelspacing=0.22, handlelength=1.6, handletextpad=0.4)

    axes[0].set_ylim(-9, 60)
    fig.tight_layout(pad=0.3)
    if save:
        fig.savefig(save, dpi=600); print("saved", save)
    plt.show()

    below = int((runs < exact[-1]).sum())
    print("\npi_psi settles at %.2f, above %d of the %d baseline runs"
          % (exact[-1], below, len(runs)))
    hit = next(r for r in c if r["exact"] >= 0.999 * TC["pool"]["oracle_topv"])
    print("g_phi within 0.1%% of a perfect ordering at epoch %d (%.0f s)"
          % (hit["epoch"], hit["elapsed"]))

fig_training()

## Fig. 3 — QoS of the returned configuration

Each method is shown at its **best of ten restarts** at $N_{pop} = N_{gen} =
120$, because on this scenario the spread between restarts is wider than the
spread between methods; a single run per method would mostly plot seed noise.

Panel (b) is worth drawing only because the screened-but-unseeded configuration
is present. With just the baseline and the full pipeline every optimised
configuration sits on the $10^{-10}$ reporting floor and the panel separates
nothing.

In [ ]:
# --- Cell 5: Fig. 3 ----------------------------------------------------------
def fig_qos(save="fig3_qos_methods.png"):
    acs = QM["ac"]
    eps = np.array(QM["eps"], float)
    x = np.arange(len(acs))
    w = 0.20
    # Log bars need a floor below the data. The default table puts AC2 at 1e-16,
    # thirteen decades below anything else here; giving the axis room for it
    # would compress every comparison that matters into the top fifth of the
    # frame, so the floor sits just below the optimised values instead and the
    # bars that fall through it are declared on the panel.
    LO_A, LO_B = 1e-11, 1e-12
    series = [("default", C_DEF, "Default EDCA")] + \
             [(k, STYLE[k]["c"], LAB_S[k]) for k in ORDER]

    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.35))
    for i, (k, c, lab) in enumerate(series):
        off = (i - 1.5) * w
        va = np.maximum(np.array(QM["configs"][k]["violation"], float), LO_A)
        pl = np.maximum(np.array(QM["configs"][k]["p_loss"], float), LO_B)
        axes[0].bar(x + off, va, w, color=c, edgecolor="black", lw=0.4,
                    label=lab, zorder=3)
        axes[1].bar(x + off, pl, w, color=c, edgecolor="black", lw=0.4,
                    label=lab, zorder=3)

    ax = axes[0]
    ax.plot(x, eps, "_", color=RED, ms=15, mew=1.4, ls="none", zorder=5,
            label=r"Target $\varepsilon_i$")
    ax.set_ylim(LO_A, 1e3)
    ax.set_ylabel(r"$\Pr(D \geq D_{\max})$")
    ax.set_title("(a) Delay violation probability")
    ax.text(0.02, 0.96, r"default-table bars below $10^{-11}$ are clipped",
            transform=ax.transAxes, va="top", fontsize=PT_NOTE, color=GREY)

    ax = axes[1]
    ax.axhline(1e-10, color=GREY, lw=0.7, ls=(0, (3, 2)), zorder=4)
    ax.text(0.02, 0.96, r"dashed: reporting floor $10^{-10}$",
            transform=ax.transAxes, va="top", fontsize=PT_NOTE, color=GREY)
    ax.set_ylim(LO_B, 1e2)
    ax.set_ylabel(r"$P_{\mathrm{loss}}$")
    ax.set_title("(b) Packet loss probability")

    for ax in axes:
        ax.set_yscale("log")
        ax.set_xticks(x); ax.set_xticklabels(acs)
        ax.set_xlabel("Access category")
        ax.grid(True, axis="y", which="major", lw=0.35, color="#D5D5D5")

    fig.tight_layout(pad=0.3)
    legend_below(fig, axes[0], ncol=5, y=-0.17)
    if save:
        fig.savefig(save, dpi=600); print("saved", save)
    plt.show()

    print("\nfeasible at every category?")
    for k, _, lab in series:
        v = np.array(QM["configs"][k]["violation"], float)
        bad = [acs[i] for i in range(len(acs)) if v[i] >= eps[i]]
        print("  %-14s %s" % (k, "yes" if not bad else "no -- " + ", ".join(bad)))

fig_qos()

## Fig. 4 — behaviour when the tightest threshold moves

**Budgets are not equal, deliberately, and the legend says so.** The same
$(N_{pop}, N_{gen})$ costs the three methods very different amounts of the
scarce quantity — calls to the analytical model. So:

- the **baseline** runs at the configuration that reproduces the objective
  reported for it in the reference work, about $35$ at $\varepsilon_1 =
  10^{-4}$; ours reaches $40.55$ there, slightly in its favour;
- each **network** runs at the largest configuration measured and *still* spends
  fewer exact calls — 1,065 and 478 against 3,362 — because a screened candidate
  is cheaper than an exactly evaluated one.

The quality ordering and the cost ordering therefore run opposite ways, and both
are on the figure.

Each curve is the **mean over 40 restarts**, with $\pm$ one standard deviation
shaded in (a). At ten restarts the standard error of each mean is 1–3 objective
points, enough to produce dips that are not there — the baseline had one.

**Panel (a) is bounded, panel (b) is not.** $F = \sum_i -\log_{10}
P_{\mathrm{loss},i}$ does not contain $\varepsilon$, which enters only through
the constraints, so relaxing $\varepsilon_1$ only enlarges the feasible set and
$F^\star$ cannot fall. That bound is on the *optimum*, which the best-of-$N$
estimates — not on the mean plotted here, which is a property of the solver.
$\sum_i \theta_i$ is not the objective at all, so nothing constrains panel (b).

In [ ]:
# --- Cell 6: Fig. 4 ----------------------------------------------------------
# Budget per method. Change these to any tag present in BM["methods"][k]["budget"]
# to see a different comparison; set all three to "120x120" for the equal
# (Npop, Ngen) version, where the baseline gets 26x the pipeline's exact calls.
SWEEP_BUDGET = {"ga": "60x60r40", "ga_gnn": "120x120gr40",
                "ga_gnn_policy": "120x120r40"}

def fig_sweep(budgets=None, save="fig4_sweep_methods.png"):
    budgets = budgets or SWEEP_BUDGET
    # Index within each method's own budget list, not by position in the global
    # one: the lists have diverged since some budgets were measured for a single
    # method only.
    idx = {k: BM["methods"][k]["budget"].index(v) for k, v in budgets.items()}
    eps1 = np.array(BM["eps1"], float)
    tgt = np.array(BM["target_sum_theta"], float)

    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.30))
    for k in ORDER:
        st, e, b = STYLE[k], BM["methods"][k], idx[k]
        lab = "%s, %s calls" % (LAB_S[k], "{:,}".format(round(e["calls"][b])))
        # Band in (a) only: in (b) the quantity is not the objective, the three
        # curves run within two points of each other, and three overlapping
        # bands would hide both them and the target line.
        for ax, mu, sd in ((axes[0], e["mean"][b], e["std"][b]),
                           (axes[1], e["sum_theta_mean"][b], None)):
            mu = np.array(mu, float)
            if sd is not None:
                sd = np.array(sd, float)
                ax.fill_between(eps1, mu - sd, mu + sd, color=st["c"],
                                alpha=0.11, lw=0, zorder=1)
            ax.plot(eps1, mu, ls=st["ls"], marker=st["m"], color=st["c"],
                    ms=4.0, mec="black", mew=0.4, zorder=3, label=lab)

    ax = axes[0]
    ax.axhline(50, color=GREY, lw=0.6, ls=(0, (1, 2)), zorder=2)
    # Left, not right: the pipeline curve reaches the ceiling from about 1e-7
    # onwards and a right-hand label lands on it.
    ax.text(0.02, 49.4, "ceiling $F = 50$", transform=ax.get_yaxis_transform(),
            ha="left", va="top", fontsize=PT_NOTE, color=GREY)
    ax.set_ylabel("Objective $F$")
    n_r = BM["methods"]["ga"]["n_restart"][idx["ga"]]
    ax.set_title("(a) Fitness value, mean of $%d$ restarts" % n_r)

    ax = axes[1]
    ax.plot(eps1, tgt, ":", color="black", lw=1.0, label=r"Target $\varepsilon$")
    ax.set_ylabel(r"$\sum_i \theta_i$")
    ax.set_title("(b) Sum of reliability indices")

    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlabel(r"$\varepsilon_1$")

    fig.tight_layout(pad=0.3)
    h, l = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    h.append(h2[-1]); l.append(l2[-1])
    legend_below(fig, axes[0], ncol=4, y=-0.16, handles=h, labels=l)
    if save:
        fig.savefig(save, dpi=600); print("saved", save)
    plt.show()

    # The bound is on the optimum, which best-of-N estimates, not on the mean.
    mono = lambda f: all(f[i] <= f[i+1] + 1e-9 for i in range(len(f) - 1))
    print()
    for k in ORDER:
        e, b = BM["methods"][k], idx[k]
        mu, sd, bs = e["mean"][b], e["std"][b], e["best"][b]
        print("  %-14s %-13s %5.0f calls, n=%d"
              % (k, budgets[k], e["calls"][b], e["n_restart"][b]))
        print("  %-14s mean %s  (sd %.2f-%.2f)  non-decreasing %s"
              % ("", " ".join("%5.2f" % v for v in mu), min(sd), max(sd), mono(mu)))
        print("  %-14s best %s  non-decreasing %s"
              % ("", " ".join("%5.2f" % v for v in bs), mono(bs)))
    M = {k: np.array(BM["methods"][k]["mean"][idx[k]]) for k in ORDER}
    strict = ((M["ga_gnn_policy"] > M["ga_gnn"]) & (M["ga_gnn"] > M["ga"])).sum()
    print("\n  pi_psi > g_phi > baseline at %d of 5 thresholds" % strict)

fig_sweep()

## Table I — cost and quality

Two comparisons, and they answer different questions.

**Upper block — one search configuration.** Here $g_\phi$ is *not* expected to
raise the objective: it replaces an exact evaluator with an approximate one, so
at an equal number of candidates examined it can at best match the baseline, and
it carries prediction error besides. What it buys is that each candidate costs
less. That is exactly what the rows show — $34.63$ against $35.04$, for
$19.8\times$ fewer exact calls.

**Lower block — one wall-clock budget.** This is where a cheaper evaluation has
to prove it buys something the optimiser can use. It does: over twenty seeds the
ordering $\pi_\psi > g_\phi > \text{baseline}$ holds at every one of the 40
wall-clock and 48 exact-call budgets measured, without exception.

No early stopping in the lower block, so each run continues to the end of its
budget; that is why the baseline's totals there exceed the upper block's.

In [ ]:
# --- Cell 7: Table I ---------------------------------------------------------
try:
    import pandas as pd
except ImportError:
    pd = None

SUCCESS = AT["success"]
BUDGETS = (1.0, 3.0, 10.0, 30.0, 100.0, 250.0)

def tables():
    # ---- upper: one search configuration, medians over 10 seeds -----------
    i200 = B["pops"].index(200)
    rows = []
    for k in ORDER:
        e = B["sweep"][k][i200]
        rows.append({"Configuration": LAB_TXT[k], "_tex": LAB_TEX[k],
                     "Time (s)": round(e["wall_med"], 1),
                     "Exact calls": int(e["eval_med"]),
                     "F": round(e["fit_med"], 2),
                     "Success": "%d/10" % sum(r["fitness"] >= SUCCESS
                                              for r in e["runs"]),
                     "Feas. init.": "%.0f%%" % (100 * np.median(
                         [r["frac_feasible_init"] for r in e["runs"]]))})
    rows.append({"Configuration": "Pipeline, Ngen = 1",
                 "_tex": r"Pipeline, $\Ngen = 1$",
                 "Time (s)": round(float(np.median(NG1["wall"])), 1),
                 "Exact calls": int(np.median(NG1["eval"])),
                 "F": round(float(np.median(NG1["fit"])), 2),
                 "Success": "%d/10" % sum(v >= SUCCESS for v in NG1["fit"]),
                 "Feas. init.": "33%"})

    # ---- lower: one wall-clock budget, mean over 20 seeds ------------------
    t = np.array(AT["t_grid"], float)
    grid = []
    for want in BUDGETS:
        i = int(np.argmin(np.abs(t - want)))
        r = {"Budget (s)": float("%.3g" % t[i])}
        for k in ORDER:
            mu = AT["methods"][k]["wall"]["mean"][i]
            r[LAB_TXT[k].strip()] = "--" if mu != mu else round(mu, 2)
        grid.append(r)

    def show(title, data):
        print("\n" + title); print("-" * len(title))
        vis = [{k: v for k, v in r.items() if not k.startswith("_")}
               for r in data]
        if pd is not None:
            df = pd.DataFrame(vis)
            try:
                from IPython.display import display; display(df)
            except ImportError:
                print(df.to_string(index=False))
            return df
        for r in vis:
            print("  " + "  ".join("%s=%s" % kv for kv in r.items()))

    t1 = show("Upper: one search configuration (Npop = 200, Ngen = 300), "
              "median of 10 seeds", rows)
    t2 = show("Lower: one wall-clock budget, mean over %d seeds, no early "
              "stopping" % AT["n_seed"], grid)
    print("\nTraining, paid once for all scenarios: 159 s training set, "
          "401 s g_phi, 81 s pi_psi -> 641 s")
    return rows, t1, t2

ROWS, T1, T2 = tables()

In [ ]:
# --- Cell 8: the upper block as LaTeX, ready to paste ------------------------
# Written field by field rather than through DataFrame.to_latex: that helper
# escapes the backslashes in the method names into \textbackslash and prints
# every float at six decimals.
def latex_table(rows):
    L = [r"\begin{table}[!t]", r"\centering",
         r"\caption{Cost of one solve, medians over $10$ seeds.}",
         r"\label{tab:cost}", r"\small",
         r"\setlength{\tabcolsep}{3pt}",
         r"\begin{tabular}{lccccc}", r"\toprule",
         r"Configuration & Time (s) & Exact calls & $F$ & Success "
         r"& Feas.\ init. \\", r"\midrule"]
    for i, r in enumerate(rows):
        if i == len(rows) - 1:
            L.append(r"\midrule")
        L.append("%s & $%.1f$ & $%s$ & $%.2f$ & $%s$ & $%s$ \\\\"
                 % (r["_tex"], r["Time (s)"],
                    "{:,}".format(r["Exact calls"]).replace(",", "{,}"),
                    r["F"], r["Success"], r["Feas. init."].replace("%", r"\%")))
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)

print(latex_table(ROWS))

## Extra — objective against compute

Not in the manuscript; its numbers are the lower block of Table I. Kept here
because the curve makes the point that the table only states: the ordering holds
across the whole budget range, not at one chosen point.

Note the honest limit. $g_\phi$ alone settles at $43.01$ and rises no further —
the surrogate's error caps what screening by it can select — while the baseline
keeps climbing and would cross that ceiling given time beyond the range
measured. Only $\pi_\psi$ lifts the ceiling.

In [ ]:
# --- Cell 9: objective against compute (not in the paper) --------------------
def fig_anytime(save="extra_anytime.png"):
    t = np.array(AT["t_grid"], float)
    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.45))

    for k in ORDER:
        st, a = STYLE[k], AT["methods"][k]["wall"]
        mu = np.array(a["mean"], float)
        sd = np.array(a["std"], float)
        su = np.array(a["success"], float)
        # Before a method's first generation there is nothing to report. The
        # run-out is NaN rather than extrapolated backwards -- extrapolating
        # there would hand free credit to whichever method starts slowest.
        ok = np.isfinite(mu)
        axes[0].fill_between(t[ok], (mu - sd)[ok], (mu + sd)[ok], color=st["c"],
                             alpha=0.10, lw=0, zorder=1)
        axes[0].plot(t[ok], mu[ok], ls=st["ls"], color=st["c"], lw=1.3,
                     zorder=3, label=LAB[k])
        axes[1].plot(t[ok], 100 * su[ok], ls=st["ls"], color=st["c"], lw=1.3,
                     zorder=3, label=LAB[k])

    ax = axes[0]
    ax.axhline(SUCCESS, color=RED, lw=0.8, ls=(0, (3, 2)), zorder=2)
    ax.text(0.02, SUCCESS - 1.5, "success threshold $F \\geq %.2f$" % SUCCESS,
            transform=ax.get_yaxis_transform(), ha="left", va="top",
            fontsize=PT_NOTE, color=RED)
    ax.set_ylabel("Objective $F$ reached by then")
    ax.set_title("(a) What each method has reached, mean of $%d$ seeds"
                 % AT["n_seed"])

    ax = axes[1]
    ax.set_ylim(-4, 108)
    ax.set_ylabel(r"Seeds reaching $F \geq %.2f$ (\%%)" % SUCCESS)
    ax.set_title("(b) How often, at the same budget")

    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlabel("Wall-clock budget per solve (s)")

    fig.tight_layout(pad=0.3)
    legend_below(fig, axes[0], ncol=3, y=-0.17)
    if save:
        fig.savefig(save, dpi=600); print("saved", save)
    plt.show()

    for axis, gk in (("wall", "t_grid"), ("evals", "e_grid")):
        g = np.array(AT[gk], float)
        a = np.array(AT["methods"]["ga_gnn"][axis]["mean"], float)
        b = np.array(AT["methods"]["ga"][axis]["mean"], float)
        c = np.array(AT["methods"]["ga_gnn_policy"][axis]["mean"], float)
        m = np.isfinite(a) & np.isfinite(b) & np.isfinite(c)
        print("  %-6s pipeline > g_phi > GA at %d of %d shared budgets "
              "(%.3g to %.3g)"
              % (axis, int(((c > a) & (a > b) & m).sum()), int(m.sum()),
                 g[m][0], g[m][-1]))

fig_anytime()